In [1]:
from datasets import load_dataset

ds = load_dataset("EdinburghNLP/xsum")

Generating test split: 100%|██████████| 11334/11334 [00:00<00:00, 52769.84 examples/s]


In [6]:
pip install evaluate

Note: you may need to restart the kernel to use updated packages.


In [7]:
import nltk
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from evaluate import load as load_metric
import torch

In [3]:
nltk.data.find('tokenizers/punkt')

FileSystemPathPointer('C:\\Users\\Gapba\\AppData\\Roaming\\nltk_data\\tokenizers\\punkt')

In [4]:
SAMPLE_SIZE = 5
data_sample = ds['test'].select(range(SAMPLE_SIZE))

In [5]:
documents = data_sample['document']
reference_summaries = data_sample['summary']

In [6]:
def lead_3_summarizer(text):
    sentences = nltk.sent_tokenize(text)
    summary = " ".join(sentences[:3])
    return summary

In [7]:
def textrank_summarizer(text, num_sentences=2):
    parser = PlaintextParser.from_string(text, SumyTokenizer("english"))
    summarizer = TextRankSummarizer()
    summary_sentences = summarizer(parser.document, num_sentences)
    summary = " ".join([str(sentence) for sentence in summary_sentences])
    return summary

In [8]:
device = 0 if torch.cuda.is_available() else -1

In [9]:
bart_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
if device != -1:
    bart_model = bart_model.to(device)

In [10]:
pegasus_tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")
pegasus_model = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-xsum")
if device != -1:
    pegasus_model = pegasus_model.to(device)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
def transformer_summarize(text, model, tokenizer, max_length=60, min_length=10):
    model_device = model.device
    
    tokenizer_input_max_length = model.config.max_position_embeddings

    inputs = tokenizer(
        text,
        max_length=tokenizer_input_max_length,
        return_tensors="pt",
        truncation=True,
        padding="longest"
    )
    
    input_ids = inputs.input_ids.to(model_device)
    attention_mask = inputs.attention_mask.to(model_device)

    summary_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        num_beams=4,
        max_length=max_length,
        min_length=min_length,
        early_stopping=True
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [12]:
generated_summaries = {
    "Lead-3": [],
    "TextRank": [],
    "BART": [],
    "PEGASUS": []
}

In [13]:
for i, doc in enumerate(documents):
    print(f"Обработка документа {i+1}/{SAMPLE_SIZE}...")
    generated_summaries["Lead-3"].append(lead_3_summarizer(doc))
    generated_summaries["TextRank"].append(textrank_summarizer(doc))
    
    print(f"  Генерация BART для документа {i+1}...")
    generated_summaries["BART"].append(transformer_summarize(doc, bart_model, bart_tokenizer))
    
    print(f"  Генерация PEGASUS для документа {i+1}...")
    generated_summaries["PEGASUS"].append(transformer_summarize(doc, pegasus_model, pegasus_tokenizer))

Обработка документа 1/5...
  Генерация BART для документа 1...
  Генерация PEGASUS для документа 1...
Обработка документа 2/5...
  Генерация BART для документа 2...
  Генерация PEGASUS для документа 2...
Обработка документа 3/5...
  Генерация BART для документа 3...
  Генерация PEGASUS для документа 3...
Обработка документа 4/5...
  Генерация BART для документа 4...
  Генерация PEGASUS для документа 4...
Обработка документа 5/5...
  Генерация BART для документа 5...
  Генерация PEGASUS для документа 5...


In [14]:
rouge_metric = load_metric("rouge")
bertscore_metric = load_metric("bertscore")

results = {}

In [15]:
print(f"Mетод Lead-3")

rouge_scores = rouge_metric.compute(predictions=generated_summaries['Lead-3'], references=reference_summaries)
print("ROUGE Scores:")
for key, value in rouge_scores.items():
    print(f"  {key}: {value*100:.2f}")

bertscore_device = "cuda:0" if device == 0 else "cpu"
bert_scores = bertscore_metric.compute(predictions=generated_summaries['Lead-3'], references=reference_summaries, lang="en", device=bertscore_device)
print("BERTScore (средние значения):")
avg_precision = sum(bert_scores['precision']) / len(bert_scores['precision'])
avg_recall = sum(bert_scores['recall']) / len(bert_scores['recall'])
avg_f1 = sum(bert_scores['f1']) / len(bert_scores['f1'])
print(f"  Precision: {avg_precision*100:.2f}")
print(f"  Recall: {avg_recall*100:.2f}")
print(f"  F1 Score: {avg_f1*100:.2f}")

Mетод Lead-3
ROUGE Scores:
  rouge1: 20.00
  rouge2: 4.11
  rougeL: 12.62
  rougeLsum: 12.78


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore (средние значения):
  Precision: 84.15
  Recall: 88.12
  F1 Score: 86.09


In [16]:
print(f"BART")

rouge_scores = rouge_metric.compute(predictions=generated_summaries['BART'], references=reference_summaries)
print("ROUGE Scores:")
for key, value in rouge_scores.items():
    print(f"  {key}: {value*100:.2f}")

bertscore_device = "cuda:0" if device == 0 else "cpu"
bert_scores = bertscore_metric.compute(predictions=generated_summaries['BART'], references=reference_summaries, lang="en", device=bertscore_device)
print("BERTScore (средние значения):")
avg_precision = sum(bert_scores['precision']) / len(bert_scores['precision'])
avg_recall = sum(bert_scores['recall']) / len(bert_scores['recall'])
avg_f1 = sum(bert_scores['f1']) / len(bert_scores['f1'])
print(f"  Precision: {avg_precision*100:.2f}")
print(f"  Recall: {avg_recall*100:.2f}")
print(f"  F1 Score: {avg_f1*100:.2f}")

BART
ROUGE Scores:
  rouge1: 24.54
  rouge2: 4.22
  rougeL: 14.27
  rougeLsum: 14.27
BERTScore (средние значения):
  Precision: 84.89
  Recall: 87.68
  F1 Score: 86.26


In [17]:
print(f"Mетод TextRank")

rouge_scores = rouge_metric.compute(predictions=generated_summaries['TextRank'], references=reference_summaries)
print("ROUGE Scores:")
for key, value in rouge_scores.items():
    print(f"  {key}: {value*100:.2f}")

bertscore_device = "cuda:0" if device == 0 else "cpu"
bert_scores = bertscore_metric.compute(predictions=generated_summaries['TextRank'], references=reference_summaries, lang="en", device=bertscore_device)
print("BERTScore (средние значения):")
avg_precision = sum(bert_scores['precision']) / len(bert_scores['precision'])
avg_recall = sum(bert_scores['recall']) / len(bert_scores['recall'])
avg_f1 = sum(bert_scores['f1']) / len(bert_scores['f1'])
print(f"  Precision: {avg_precision*100:.2f}")
print(f"  Recall: {avg_recall*100:.2f}")
print(f"  F1 Score: {avg_f1*100:.2f}")

Mетод TextRank
ROUGE Scores:
  rouge1: 14.89
  rouge2: 2.70
  rougeL: 10.18
  rougeLsum: 10.14
BERTScore (средние значения):
  Precision: 83.71
  Recall: 87.86
  F1 Score: 85.73


In [18]:
print(f"Mетод PEGASUS")

rouge_scores = rouge_metric.compute(predictions=generated_summaries['PEGASUS'], references=reference_summaries)
print("ROUGE Scores:")
for key, value in rouge_scores.items():
    print(f"  {key}: {value*100:.2f}")

bertscore_device = "cuda:0" if device == 0 else "cpu"
bert_scores = bertscore_metric.compute(predictions=generated_summaries['PEGASUS'], references=reference_summaries, lang="en", device=bertscore_device)
print("BERTScore (средние значения):")
avg_precision = sum(bert_scores['precision']) / len(bert_scores['precision'])
avg_recall = sum(bert_scores['recall']) / len(bert_scores['recall'])
avg_f1 = sum(bert_scores['f1']) / len(bert_scores['f1'])
print(f"  Precision: {avg_precision*100:.2f}")
print(f"  Recall: {avg_recall*100:.2f}")
print(f"  F1 Score: {avg_f1*100:.2f}")

Mетод PEGASUS
ROUGE Scores:
  rouge1: 59.56
  rouge2: 34.33
  rougeL: 49.82
  rougeLsum: 50.28
BERTScore (средние значения):
  Precision: 93.25
  Recall: 92.72
  F1 Score: 92.98


In [19]:
print(f"ОРИГИНАЛЬНЫЙ ДОКУМЕНТ (начало):\n{documents[0][:500]}...\n")
print(f"ЭТАЛОННОЕ РЕЗЮМЕ:\n{reference_summaries[0]}\n")

for method_name, summaries_list in generated_summaries.items():
    if summaries_list: # Проверка, что список не пуст
            print(f"РЕЗЮМЕ ({method_name}):\n{summaries_list[0]}\n")

ОРИГИНАЛЬНЫЙ ДОКУМЕНТ (начало):
Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than ever were getting help to address housing problems.
Changes to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.
Prison Link C...

ЭТАЛОННОЕ РЕЗЮМЕ:
There is a "chronic" need for more housing for prison leavers in Wales, according to a charity.

РЕЗЮМЕ (Lead-3):
Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation. Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders. The Welsh Government said more people than ever were getting help to addr

In [20]:
print("\n--- Сводные результаты (ROUGE-1 F1, BERTScore F1) ---")
print(f"{'Метод':<15} | {'ROUGE-1 F1':<12} | {'BERTScore F1':<12}")
print()
for method, scores in results.items():
    rouge1_f1 = scores["rouge"].get('rouge1', 0.0) * 100 
    bert_f1 = scores["bertscore_f1"] * 100
    print(f"{method:<15} | {rouge1_f1:<12.2f} | {bert_f1:<12.2f}")


--- Сводные результаты (ROUGE-1 F1, BERTScore F1) ---
Метод           | ROUGE-1 F1   | BERTScore F1

